In [18]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

verbose = 0

data_path_list = sorted(
    list(Path("../validate").glob("*.csv")),
    key=lambda p: p.stat().st_ctime,
)
basis_args = "cc-pVDZ"
print(basis_args)

with open("../cc2cc/utils/g2.json") as f:
    json_data = json.load(f)

# accumulate summary dictionaries for each file
summary_list = []

for data_path in data_path_list:
    data = pd.read_csv(data_path)

    data["name"] = data["name"].str.split(f"_{basis_args}").str[0]

    data_name = []
    data_atomic_energy_dft = []
    data_atomic_energy_ai = []
    data_atomic_dft_ele = []
    data_atomic_scf_ele = []
    data_atomic_dft_dip = []
    data_atomic_scf_dip = []

    for i_name in data["name"]:
        if i_name not in json_data["reaction-atomic-energy"]:
            continue
        systems_list = json_data["reaction-atomic-energy"][i_name]["systems"]
        stoichiometry_list = json_data["reaction-atomic-energy"][i_name][
            "stoichiometry"
        ]

        atomic_energy_dft = 0
        atomic_energy_ai = 0
        for i in range(len(systems_list)):
            atomic_energy_dft += data[data["name"] == systems_list[i]][
                "error_dft_ene"
            ].values[0] * int(stoichiometry_list[i])
            atomic_energy_ai += data[data["name"] == systems_list[i]][
                "error_scf_ene"
            ].values[0] * int(stoichiometry_list[i])
            if verbose == 2:
                print(
                    data[data["name"] == systems_list[i]]["error_dft_ene"].values[0],
                    int(stoichiometry_list[i]),
                    systems_list[i],
                )
        data_atomic_energy_dft.append(atomic_energy_dft)
        data_atomic_energy_ai.append(atomic_energy_ai)
        data_name.append(i_name)

        data_atomic_dft_ele.append(
            data.loc[data["name"] == i_name, "error_dft_ele"].values[0]
        )
        data_atomic_scf_ele.append(
            data.loc[data["name"] == i_name, "error_scf_ele"].values[0]
        )
        data_atomic_dft_dip.append(
            data.loc[data["name"] == i_name, "error_dft_dip"].values[0]
        )
        data_atomic_scf_dip.append(
            data.loc[data["name"] == i_name, "error_scf_dip"].values[0]
        )

    data_name = np.array(data_name)
    data_energy_ai = np.array(data["error_scf_ene"])
    data_energy_dft = np.array(data["error_dft_ene"])
    data_atomic_energy_dft = np.array(data_atomic_energy_dft)
    data_atomic_energy_ai = np.array(data_atomic_energy_ai)
    data_atomic_dft_ele = np.array(data_atomic_dft_ele)
    data_atomic_scf_ele = np.array(data_atomic_scf_ele)

    if verbose >= 1:
        sorted_indices = np.argsort(np.abs(data_atomic_energy_dft))[::-1][:10]
        sorted_data_atomic_energy_dft = data_name[sorted_indices]
        print(
            "dft",
            np.array(
                [
                    sorted_data_atomic_energy_dft,
                    np.array(data_atomic_energy_dft)[sorted_indices],
                ]
            ).T,
        )
        sorted_indices = np.argsort(np.abs(data_atomic_energy_ai))[::-1][:10]
        sorted_data_atomic_energy_ai = data_name[sorted_indices]
        print(
            "ai",
            np.array(
                [
                    sorted_data_atomic_energy_ai,
                    np.array(data_atomic_energy_ai)[sorted_indices],
                ]
            ).T,
        )
        print("dft_ele", np.mean(np.abs(data_atomic_dft_ele)))
        print("scf_ele", np.mean(np.abs(data_atomic_scf_ele)))
        print("dft_dip", np.mean(np.abs(data_atomic_dft_dip)))
        print("scf_dip", np.mean(np.abs(data_atomic_scf_dip)))

    summary = {
        "File": data_path.stem.split("_")[2],
        "AI AE": f"{np.mean(np.abs(data_atomic_energy_ai)):.2f}",
        "DFT AE": f"{np.mean(np.abs(data_atomic_energy_dft)):.2f}",
        "AI |E|": f"{np.mean(np.abs(data_energy_ai)):.2f}",
        "DFT |E|": f"{np.mean(np.abs(data_energy_dft)):.2f}",
        "AI Ele": f"{np.mean(np.abs(data_atomic_scf_ele)):.2f}",
        "DFT Ele": f"{np.mean(np.abs(data_atomic_dft_ele)):.2f}",
        "AI Dip": f"{np.mean(np.abs(data_atomic_scf_dip)):.3f}",
        "DFT Dip": f"{np.mean(np.abs(data_atomic_dft_dip)):.3f}",
        "Processed": f"{len(data_atomic_energy_dft)} / {len(json_data["reaction-atomic-energy"])}",
    }
    summary_list.append(summary)
    print()  # blank line between iterations

# display one summary table for all files
df_summary = pd.DataFrame(summary_list)
print("Summary of Atomic Energies:")
display(df_summary)

cc-pVDZ






Summary of Atomic Energies:


,File,AI AE,DFT AE,AI |E|,DFT |E|,AI Ele,DFT Ele,AI Dip,DFT Dip,Processed
0,atom-1-3885473,2.17,29.83,2.39,320.80,0.16,0.16,0.031,0.023,140 / 140
1,atom-1-2963972,1.26,29.83,1.40,320.80,0.12,0.16,0.026,0.023,140 / 140
2,atom-1-2995870,1.34,29.83,1.24,320.80,0.12,0.16,0.027,0.023,140 / 140
3,atom-1-3210973,2.02,29.83,2.02,320.80,0.17,0.16,0.028,0.023,140 / 140
4,atom-1-1916450,1.30,29.83,1.35,320.80,0.12,0.16,0.028,0.023,140 / 140
5,atom-1-1464870,1.84,29.83,2.16,320.80,0.12,0.16,0.028,0.023,140 / 140
